In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
print("Drive mounted successfully!")

Mounted at /content/drive
Drive mounted successfully!
Project folder found: /content/drive/MyDrive/iam_rag_project/


In [ ]:
# Cell 2: Define all project paths
import os

# ============================================
# ONLY CHANGE THIS IF YOUR PATH IS DIFFERENT
# ============================================
DRIVE_BASE = '/content/drive/MyDrive/iam_rag_project/'

# All project paths — never hardcode these elsewhere in the notebook
PATHS = {
    # Data paths
    'raw':           DRIVE_BASE + 'data/raw/',
    'processed':     DRIVE_BASE + 'data/processed/',
    'benchmark':     DRIVE_BASE + 'data/benchmark/',
    'train':         DRIVE_BASE + 'data/benchmark/train/',
    'test':          DRIVE_BASE + 'data/benchmark/test/',

    # Index paths
    'bm25':          DRIVE_BASE + 'indexes/bm25/',
    'dense':         DRIVE_BASE + 'indexes/dense/',
    'sparse':        DRIVE_BASE + 'indexes/sparse/',

    # Results
    'results':       DRIVE_BASE + 'results/',
}

# Create all directories if they don't exist yet
for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)

# Verify everything looks correct
print("Project paths:")
for name, path in PATHS.items():
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {name}: {path}")

Project paths:
  ✓ raw: /content/drive/MyDrive/iam_rag_project/data/raw/
  ✓ processed: /content/drive/MyDrive/iam_rag_project/data/processed/
  ✓ benchmark: /content/drive/MyDrive/iam_rag_project/data/benchmark/
  ✓ train: /content/drive/MyDrive/iam_rag_project/data/benchmark/train/
  ✓ test: /content/drive/MyDrive/iam_rag_project/data/benchmark/test/
  ✓ bm25: /content/drive/MyDrive/iam_rag_project/indexes/bm25/
  ✓ dense: /content/drive/MyDrive/iam_rag_project/indexes/dense/
  ✓ sparse: /content/drive/MyDrive/iam_rag_project/indexes/sparse/
  ✓ results: /content/drive/MyDrive/iam_rag_project/results/


In [ ]:
# Install all dependencies

print("Installing dependencies...")

# Core scraping and data processing
!pip install -q requests beautifulsoup4 lxml

# Data handling
!pip install -q pandas numpy

# Progress bars
!pip install -q tqdm

!pip install -q torch

print("All dependencies installed!")

Installing dependencies...
All dependencies installed!


In [ ]:
# Import all libraries and verify installation

# ── Standard library ──────────────────────────────
import os
import json
import re
import time
from pathlib import Path
from datetime import datetime
import torch

# ── Data processing ───────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm

# ── Web scraping ──────────────────────────────────
import requests
from bs4 import BeautifulSoup

print("All imports successful!")
print()

# Quick sanity checks
print("Environment check:")
print(f"  PyTorch version:    {torch.__version__}")

All imports successful!

Environment check:
  PyTorch version:    2.10.0+cpu


'\nprint(f"  CUDA available:     {torch.cuda.is_available()}")\nif torch.cuda.is_available():\n    print(f"  GPU:                {torch.cuda.get_device_name(0)}")\nprint(f"  FAISS version:      {faiss.__version__}")\nprint(f"  Transformers ready: ✓")\nprint(f"  Drive mounted:      {os.path.exists(DRIVE_BASE)}")\n'

In [ ]:
# Create updated folder structure

import os

def create_folder_structure(base_path):
    """
    Create the complete project folder structure.
    Safe to re-run — won't overwrite existing files.
    """
    folders = [
        # Raw data — one subfolder per source
        'data/raw/service_auth/',
        'data/raw/managed_policies/',

        # Processed data — one file per service
        'data/processed/by_service/',

        # Benchmark dataset
        'data/benchmark/',

        # Indexes (for later)
        'indexes/bm25/',
        'indexes/dense/',
        'indexes/sparse/',

        # Results (for later)
        'results/',
    ]

    for folder in folders:
        full_path = os.path.join(base_path, folder)
        os.makedirs(full_path, exist_ok=True)

    print("Folder structure created:")
    print()

    # Print tree view so you can verify visually
    for root, dirs, files in os.walk(base_path):
        # Skip hidden folders
        dirs[:] = [d for d in sorted(dirs) if not d.startswith('.')]
        depth = root.replace(base_path, '').count(os.sep)
        indent = '    ' * depth
        folder_name = os.path.basename(root)
        print(f"{indent}📁 {folder_name}/")

        # Show files if any exist
        sub_indent = '    ' * (depth + 1)
        for file in sorted(files):
            print(f"{sub_indent}📄 {file}")

create_folder_structure(DRIVE_BASE)

Folder structure created:

📁 /
    📄 iam_rag_project.ipynb
📁 data/
    📁 benchmark/
        📁 test/
        📁 train/
    📁 processed/
        📁 by_service/
    📁 raw/
        📁 managed_policies/
        📁 service_auth/
📁 indexes/
    📁 bm25/
    📁 dense/
    📁 sparse/
📁 results/


In [ ]:
# Cell 8: Define which services to scrape and their metadata
# Keeping this as a separate config cell makes it easy to
# add/remove services without touching scraping logic

# Each entry maps:
#   url_suffix   → the AWS docs page identifier
#   prefix       → the IAM action prefix (e.g. "s3" in "s3:GetObject")
#   display_name → human readable name for logging

# Updated SERVICES_CONFIG with correct IAM URL and additional IAM services

SERVICES_CONFIG = [
    {
        'url_suffix':   'list_amazons3',
        'prefix':       's3',
        'display_name': 'Amazon S3',
        'filename':     's3_actions.json'
    },
    {
        'url_suffix':   'list_awslambda',
        'prefix':       'lambda',
        'display_name': 'AWS Lambda',
        'filename':     'lambda_actions.json'
    },
    {
        'url_suffix':   'list_amazondynamodb',
        'prefix':       'dynamodb',
        'display_name': 'Amazon DynamoDB',
        'filename':     'dynamodb_actions.json'
    },
    {
        'url_suffix':   'list_amazonec2',
        'prefix':       'ec2',
        'display_name': 'Amazon EC2',
        'filename':     'ec2_actions.json'
    },

    # ── Fixed IAM URL ─────────────────────────────────────────
    {
        'url_suffix':   'list_awsidentityandaccessmanagementiam',  # fixed
        'prefix':       'iam',
        'display_name': 'AWS IAM',
        'filename':     'iam_actions.json'
    },

    # ── Additional IAM-related services found in index ────────
    # Worth adding since your project is specifically about IAM
    {
        'url_suffix':   'list_awsiamaccessanalyzer',
        'prefix':       'access-analyzer',
        'display_name': 'AWS IAM Access Analyzer',
        'filename':     'iam_access_analyzer_actions.json'
    },
    {
        'url_suffix':   'list_awsiamidentitycenter',
        'prefix':       'sso',
        'display_name': 'AWS IAM Identity Center',
        'filename':     'iam_identity_center_actions.json'
    },

    # ── Original remaining services ───────────────────────────
    {
        'url_suffix':   'list_amazonrds',
        'prefix':       'rds',
        'display_name': 'Amazon RDS',
        'filename':     'rds_actions.json'
    },
    {
        'url_suffix':   'list_amazonsns',
        'prefix':       'sns',
        'display_name': 'Amazon SNS',
        'filename':     'sns_actions.json'
    },
    {
        'url_suffix':   'list_amazonsqs',
        'prefix':       'sqs',
        'display_name': 'Amazon SQS',
        'filename':     'sqs_actions.json'
    },
    {
        'url_suffix':   'list_amazoncloudwatch',
        'prefix':       'cloudwatch',
        'display_name': 'Amazon CloudWatch',
        'filename':     'cloudwatch_actions.json'
    },
    {
        'url_suffix':   'list_awscloudformation',
        'prefix':       'cloudformation',
        'display_name': 'AWS CloudFormation',
        'filename':     'cloudformation_actions.json'
    },
    # ── High reference services from managed policies ─────────
    {
        'url_suffix':   'list_amazoncloudwatchlogs',
        'prefix':       'logs',
        'display_name': 'Amazon CloudWatch Logs',
        'filename':     'cloudwatch_logs_actions.json'
    },
    {
        'url_suffix':   'list_amazonelasticcontainerregistry',
        'prefix':       'ecr',
        'display_name': 'Amazon ECR',
        'filename':     'ecr_actions.json'
    },
    {
        'url_suffix':   'list_awsx-ray',
        'prefix':       'xray',
        'display_name': 'AWS X-Ray',
        'filename':     'xray_actions.json'
    },
    {
        'url_suffix':   'list_amazonelasticcontainerservice',
        'prefix':       'ecs',
        'display_name': 'Amazon ECS',
        'filename':     'ecs_actions.json'
    },
    {
        'url_suffix':   'list_awssystemsmanager',
        'prefix':       'ssm',
        'display_name': 'AWS Systems Manager',
        'filename':     'ssm_actions.json'
    },

    # ── Fundamental IAM services ──────────────────────────────
    {
        'url_suffix':   'list_awssecuritytokenservice',
        'prefix':       'sts',
        'display_name': 'AWS STS',
        'filename':     'sts_actions.json'
    },
    {
        'url_suffix':   'list_awskeymanagementservice',
        'prefix':       'kms',
        'display_name': 'AWS KMS',
        'filename':     'kms_actions.json'
    },
    {
        'url_suffix':   'list_awssecretsmanager',
        'prefix':       'secretsmanager',
        'display_name': 'AWS Secrets Manager',
        'filename':     'secretsmanager_actions.json'
    },
    # additional tricky but frequently used services
    {
        'url_suffix':   'list_amazonapigateway',
        'prefix':       'execute-api',
        'display_name': 'Amazon API Gateway',
        'filename':     'apigateway_actions.json'
    },
    {
        'url_suffix':   'list_amazonapigatewaymanagement',
        'prefix':       'apigateway',
        'display_name': 'Amazon API Gateway Management',
        'filename':     'apigateway_management_actions.json'
    },
    {
        'url_suffix':   'list_amazonapigatewaymanagementv2',
        'prefix':       'apigateway',
        'display_name': 'Amazon API Gateway Management V2',
        'filename':     'apigateway_management_v2_actions.json'
    },
    {
        'url_suffix':   'list_awscodedeploy',
        'prefix':       'codedeploy',
        'display_name': 'AWS CodeDeploy',
        'filename':     'codedeploy_actions.json'
    },
    {
        'url_suffix':   'list_amazoneventbridge',
        'prefix':       'events',
        'display_name': 'Amazon EventBridge',
        'filename':     'events_actions.json'
    },
    {
        'url_suffix':   'list_amazonapigateway',
        'prefix':       'execute-api',
        'display_name': 'Amazon API Gateway Execute API',
        'filename':     'execute_api_actions.json'
    },
    {
        'url_suffix':   'list_awsdirectoryservice',
        'prefix':       'ds',
        'display_name': 'AWS Directory Service',
        'filename':     'ds_actions.json'
    },
    {
        'url_suffix': 'list_amazonkinesisanalytics',
        'prefix': 'kinesis',
        'display_name': 'AWS Kinesis Analytics',
        'filename': 'kinesis_analytics_actions.json'
    },
    {
        'url_suffix': 'list_amazonkinesisdatastreams',
        'prefix': 'kinesis',
        'display_name': 'AWS Kinesis Data Streams',
        'filename': 'kinesis_datastreams_actions.json'
    },
    {
        'url_suffix': 'list_amazonkinesisfirehose',
        'prefix': 'firehose',
        'display_name': 'AWS Kinesis Firehose',
        'filename': 'kinesis_firehose_actions.json'
    },
    {
        'url_suffix': 'list_awsstepfunctions',
        'prefix': 'states',
        'display_name': 'AWS Step Functinos',
        'filename': 'states_actions.json'
    },
    {
        'url_suffix': 'list_awscodebuild',
        'prefix': 'codebuild',
        'display_name': 'AWS CodeBuild',
        'filename': 'codebuild_actions.json'
    },
    {
        'url_suffix': 'list_amazonathena',
        'prefix': 'athena',
        'display_name': 'AWS Athena',
        'filename': 'athena_actions.json'
    },
    {
        'url_suffix': 'list_amazonredshiftdataapi',
        'prefix': 'redshift-data',
        'display_name': 'AWS Redshift Data API',
        'filename': 'redshift_data_actions.json'
    },
    {
        'url_suffix': 'list_amazonredshift',
        'prefix': 'redshift',
        'display_name': 'AWS Redshift',
        'filename': 'redshift_actions.json'
    },
    {
        'url_suffix': 'list_awscodepipeline',
        'prefix': 'codepipeline',
        'display_name': 'AWS CodePipeline',
        'filename': 'codepipeline_actions.json'
    },
    {
        'url_suffix': 'list_awscodestarnotifications',
        'prefix': 'codestar-notifications',
        'display_name': 'AWS CodeStar Notifications',
        'filename': 'codestar_notifications_actions.json'
    },
    {
        'url_suffix': 'list_amazonmanagedstreamingforapachekafka',
        'prefix': 'kafka',
        'display_name': 'AWS Managed Streaming for Apache Kafka',
        'filename': 'kafka_actions.json'
    },
    {
        'url_suffix': 'list_awsglue',
        'prefix': 'glue',
        'display_name': 'AWS Glue',
        'filename': 'glue_actions.json'
    },
    {
        'url_suffix': '/list_amazonelasticcontainerregistrypublic',
        'prefix': 'ecr-public',
        'display_name': 'Amazon ECR Public',
        'filename': 'ecr_public_actions.json'
    },
    {
        'url_suffix': 'list_amazonelastickubernetesservice',
        'prefix': 'eks',
        'display_name': 'Amazon EKS',
        'filename': 'eks_actions.json'
    },
    {
        'url_suffix': 'list_awselasticloadbalancing',
        'prefix': 'elasticloadbalancing',
        'display_name': 'AWS Elastic Load Balancing',
        'filename': 'elb_actions.json'
    },
    {
        'url_suffix': 'list_awselasticloadbalancingv2',
        'prefix': 'elasticloadbalancing',
        'display_name': 'AWS Elastic Load Balancing V2',
        'filename': 'elb_v2_actions.json'
    },
    {
        'url_suffix': 'list_awsorganizations',
        'prefix': 'organizations',
        'display_name': 'AWS Organizations',
        'filename': 'organizations_actions.json'
    },
    {
        'url_suffix': 'list_amazoncloudwatchobservabilityaccessmanager',
        'prefix': 'oam',
        'display_name': 'AWS CloudWatch Observability Access Manager',
        'filename': 'oam_actions.json'
    },
    {
        'url_suffix': 'list_awsresourcegroups',
        'prefix': 'resource-groups',
        'display_name': 'AWS Resource Groups',
        'filename': 'resource_groups_actions.json'
    },
    {
        'url_suffix': 'list_amazons3objectlambda',
        'prefix': 's3-object-lambda',
        'display_name': 'Amazon S3 Object Lambda',
        'filename': 's3_object_lambda_actions.json'
    },
    {
        'url_suffix': 'list_amazonrekognition',
        'prefix': 'rekognition',
        'display_name': 'Amazon Rekognition',
        'filename': 'rekognition_actions.json'
    },
    {
        'url_suffix': 'list_amazonsimpleemailservice-mailmanager',
        'prefix': 'ses',
        'display_name': 'Amazon Simple Email Service (SES) - Mail Manager',
        'filename': 'ses_mailmanager_actions.json'
    },
    {
        'url_suffix': 'list_amazonsimpleemailservicev2',
        'prefix': 'ses',
        'display_name': 'Amazon Simple Email Service (SES) - V2',
        'filename': 'ses_v2_actions.json'
    },
    {
        'url_suffix': 'list_amazonredshiftserverless',
        'prefix': 'redshift-serverless',
        'display_name': 'Amazon Redshift Serverless',
        'filename': 'redshift_serverless_actions.json'
    },
     {
        'url_suffix': 'list_amazonbedrock',
        'prefix': 'bedrock',
        'display_name': 'Amazon Bedrock',
        'filename': 'bedrock_actions.json'
    },
    {
        'url_suffix': 'list_list_amazonbedrockagentcore',
        'prefix': 'bedrock-agentcore',
        'display_name': 'Amazon Bedrock Agent Core',
        'filename': 'bedrock_agentcore_actions.json'
    },
    {
        'url_suffix': 'list_awscertificatemanager',
        'prefix': 'acm',
        'display_name': 'AWS Certificate Manager',
        'filename': 'acm_actions.json'
    },
    {
        'url_suffix': 'list_awsconfig',
        'prefix': 'config',
        'display_name': 'AWS Config',
        'filename': 'config_actions.json'
    },
    {
        'url_suffix': 'list_amazonresourcegrouptaggingapi',
        'prefix': 'tag',
        'display_name': 'Amazon Resource Groups Tagging API',
        'filename': 'tag_actions.json'
    },

]

BASE_URL = (
    "https://docs.aws.amazon.com/"
    "service-authorization/latest/reference/"
)

print(f"Configured {len(SERVICES_CONFIG)} services:")
print()
for svc in SERVICES_CONFIG:
    print(f"  {svc['display_name']:<40} → {svc['filename']}")

Configured 56 services:

  Amazon S3                                → s3_actions.json
  AWS Lambda                               → lambda_actions.json
  Amazon DynamoDB                          → dynamodb_actions.json
  Amazon EC2                               → ec2_actions.json
  AWS IAM                                  → iam_actions.json
  AWS IAM Access Analyzer                  → iam_access_analyzer_actions.json
  AWS IAM Identity Center                  → iam_identity_center_actions.json
  Amazon RDS                               → rds_actions.json
  Amazon SNS                               → sns_actions.json
  Amazon SQS                               → sqs_actions.json
  Amazon CloudWatch                        → cloudwatch_actions.json
  AWS CloudFormation                       → cloudformation_actions.json
  Amazon CloudWatch Logs                   → cloudwatch_logs_actions.json
  Amazon ECR                               → ecr_actions.json
  AWS X-Ray                           

In [ ]:
# Cell 9: Scraping functions with built-in action validation

import requests
from bs4 import BeautifulSoup
import time

def is_valid_action(action_dict):
    """
    Validate that a scraped entry is a real IAM action
    and not a misidentified resource type or other table artifact.

    Real IAM actions always:
    1. Start with uppercase letter  (e.g. AcceptAddressTransfer)
    2. Do NOT end with *            (resource types end with * e.g. subnet*)
    3. Have a non-empty access_level (Read / Write / List / Tagging / etc.)
    4. Have a description           (starts with "Grants permission")
    """
    action_name  = action_dict.get('action_name', '')
    description  = action_dict.get('description', '')
    access_level = action_dict.get('access_level', '')

    if not action_name:
        return False

    if not action_name[0].isupper():
        return False

    if action_name.endswith('*'):
        return False

    if not access_level:
        return False

    if not description:
        return False

    return True


def scrape_service_actions(service_config, base_url):
    """
    Scrape all IAM actions for a single AWS service.
    Includes built-in validation to filter out misidentified
    resource type rows (fixes EC2 over-counting issue).

    Args:
        service_config: dict from SERVICES_CONFIG
        base_url:       AWS docs base URL

    Returns:
        list of validated raw action dicts
    """
    url     = base_url + service_config['url_suffix'] + '.html'
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        actions      = []
        seen_actions = set()  # prevent duplicates

        # ── Find the actions table ────────────────────────────
        tables = soup.find_all('table')

        if not tables:
            print(f"    No tables found at {url}")
            return []

        action_table = tables[0]
        rows         = action_table.find_all('tr')[1:]  # skip header

        current_action = None

        for row in rows:
            cells = row.find_all('td')
            if not cells:
                continue

            action_name_cell = cells[0].text.strip()

            if action_name_cell:
                action_key = (
                    f"{service_config['prefix']}:{action_name_cell}"
                )

                # ── Skip duplicate action names ───────────────
                if action_key in seen_actions:
                    # Still update pointer so continuation rows
                    # are attributed to the right action
                    current_action = next(
                        (a for a in actions
                         if a['action'] == action_key),
                        None
                    )
                    continue

                seen_actions.add(action_key)

                current_action = {
                    'service':           service_config['display_name'],
                    'service_prefix':    service_config['prefix'],
                    'action_name':       action_name_cell,
                    'action':            action_key,
                    'description':       cells[1].text.strip() if len(cells) > 1 else '',
                    'access_level':      cells[2].text.strip() if len(cells) > 2 else '',
                    'resource_types':    [],
                    'condition_keys':    [],
                    'dependent_actions': []
                }
                actions.append(current_action)

            # ── Add resource/condition info to current action ─
            if current_action:
                if len(cells) > 3:
                    resource = cells[3].text.strip()
                    if resource and resource not in current_action['resource_types']:
                        current_action['resource_types'].append(resource)

                if len(cells) > 4:
                    conditions = [
                        c.strip() for c in cells[4].text.split('\n')
                        if c.strip()
                    ]
                    for c in conditions:
                        if c not in current_action['condition_keys']:
                            current_action['condition_keys'].append(c)

                if len(cells) > 5:
                    dependent = [
                        d.strip() for d in cells[5].text.split('\n')
                        if d.strip()
                    ]
                    for d in dependent:
                        if d not in current_action['dependent_actions']:
                            current_action['dependent_actions'].append(d)

        # ── Validate all actions before returning ─────────────
        # This filters out misidentified resource type rows
        # (fixes the EC2 1,269 → ~400-500 over-counting issue)
        valid_actions   = [a for a in actions if is_valid_action(a)]
        invalid_count   = len(actions) - len(valid_actions)

        if invalid_count > 0:
            print(f"    Filtered out {invalid_count} invalid entries "
                  f"(resource types / empty rows)")

        return valid_actions

    except requests.exceptions.RequestException as e:
        print(f"    Request error: {e}")
        return []
    except Exception as e:
        print(f"    Parsing error: {e}")
        return []


def process_raw_actions_for_service(raw_actions, service_config):
    """
    Convert raw scraped actions into clean structured documents.
    One document per action — mirrors Gorilla's JSON format.

    Args:
        raw_actions:    list of validated dicts from scrape_service_actions()
        service_config: dict from SERVICES_CONFIG

    Returns:
        dict with service metadata and list of action documents
    """
    documents = []

    for action in raw_actions:
        # Skip actions with no description (should be caught by
        # is_valid_action already, but defensive check here too)
        if not action.get('description'):
            continue

        doc = {
            # ── Identity ──────────────────────────────────────
            'doc_id':         f"{action['service_prefix']}_{action['action_name']}".lower(),
            'action':         action['action'],
            'action_name':    action['action_name'],
            'service':        action['service'],
            'service_prefix': action['service_prefix'],

            # ── Semantic fields (used for retrieval) ──────────
            'description':    action['description'],
            'access_level':   action['access_level'],

            # ── Policy generation fields ──────────────────────
            'resource_types':    action['resource_types'],
            'condition_keys':    action['condition_keys'],
            'dependent_actions': action['dependent_actions'],

            # ── Combined text for embedding ───────────────────
            # All fields concatenated into one searchable string
            # This is what gets embedded and searched during retrieval
            'full_text': (
                f"IAM Action: {action['action']}. "
                f"Service: {action['service']}. "
                f"Description: {action['description']}. "
                f"Access level: {action['access_level']}. "
                f"Applicable resource types: "
                f"{', '.join(action['resource_types']) or 'All resources'}. "
                f"Supported condition keys: "
                f"{', '.join(action['condition_keys']) or 'None'}."
            )
        }
        documents.append(doc)

    return {
        'service':        service_config['display_name'],
        'service_prefix': service_config['prefix'],
        'action_count':   len(documents),
        'actions':        documents
    }


print("✓ is_valid_action() defined")
print("✓ scrape_service_actions() defined  — includes validation filter")
print("✓ process_raw_actions_for_service() defined")

✓ is_valid_action() defined
✓ scrape_service_actions() defined  — includes validation filter
✓ process_raw_actions_for_service() defined


In [ ]:
# Cell 10: Execute scraping and save one file per service

print("=" * 55)
print("SCRAPING SERVICE AUTHORIZATION REFERENCE")
print("=" * 55)
print()

# Track results across all services
scraping_summary = []

for service_config in SERVICES_CONFIG:
    print(f"Scraping {service_config['display_name']}...")

    # ── Step 1: Scrape raw data ───────────────────────────────
    raw_actions = scrape_service_actions(service_config, BASE_URL)

    if not raw_actions:
        print(f"  ✗ No actions found — skipping\n")
        scraping_summary.append({
            'service':  service_config['display_name'],
            'status':   'failed',
            'count':    0
        })
        continue

    # ── Step 2: Save raw data ─────────────────────────────────
    raw_filename = service_config['filename'].replace(
        '_actions.json', '_actions_raw.json'
    )
    raw_path = os.path.join(
        DRIVE_BASE, 'data/raw/service_auth/', raw_filename
    )
    with open(raw_path, 'w') as f:
        json.dump(raw_actions, f, indent=2)

    # ── Step 3: Process into structured documents ─────────────
    processed_data = process_raw_actions_for_service(
        raw_actions, service_config
    )

    # ── Step 4: Save processed — one file per service ─────────
    processed_path = os.path.join(
        DRIVE_BASE, 'data/processed/by_service/',
        service_config['filename']
    )
    with open(processed_path, 'w') as f:
        json.dump(processed_data, f, indent=2)

    # ── Log result ────────────────────────────────────────────
    print(f"  ✓ {processed_data['action_count']} actions scraped and saved")
    print(f"    Raw:       {raw_path}")
    print(f"    Processed: {processed_path}")
    print()

    scraping_summary.append({
        'service': service_config['display_name'],
        'status':  'success',
        'count':   processed_data['action_count'],
        'file':    service_config['filename']
    })

    # Be polite to AWS servers
    time.sleep(1)

# Print summary
print("=" * 55)
print("SCRAPING SUMMARY")
print("=" * 55)
total_actions = 0
for entry in scraping_summary:
    status = "✓" if entry['status'] == 'success' else "✗"
    count  = entry['count']
    total_actions += count
    print(f"  {status} {entry['service']:<30} {count:>4} actions")

print(f"\n  Total actions scraped: {total_actions}")

SCRAPING SERVICE AUTHORIZATION REFERENCE

Scraping Amazon S3...
    Filtered out 6 invalid entries (resource types / empty rows)
  ✓ 168 actions scraped and saved
    Raw:       /content/drive/MyDrive/iam_rag_project/data/raw/service_auth/s3_actions_raw.json
    Processed: /content/drive/MyDrive/iam_rag_project/data/processed/by_service/s3_actions.json

Scraping AWS Lambda...
    Filtered out 5 invalid entries (resource types / empty rows)
  ✓ 88 actions scraped and saved
    Raw:       /content/drive/MyDrive/iam_rag_project/data/raw/service_auth/lambda_actions_raw.json
    Processed: /content/drive/MyDrive/iam_rag_project/data/processed/by_service/lambda_actions.json

Scraping Amazon DynamoDB...
    Filtered out 3 invalid entries (resource types / empty rows)
  ✓ 78 actions scraped and saved
    Raw:       /content/drive/MyDrive/iam_rag_project/data/raw/service_auth/dynamodb_actions_raw.json
    Processed: /content/drive/MyDrive/iam_rag_project/data/processed/by_service/dynamodb_actio

In [ ]:
# Cell 11: Build combined action_documents.json and ground_truth_actions.json
# These are built BY READING the per-service files — not scraped again

print("Building combined files from per-service data...")
print()

all_documents  = []
ground_truth   = {
    'valid_actions':        [],
    'action_to_resources':  {},
    'action_to_conditions': {},
    'action_details':       {}
}

# Read each per-service file and combine
by_service_path = os.path.join(DRIVE_BASE, 'data/processed/by_service/')
service_files   = sorted([
    f for f in os.listdir(by_service_path)
    if f.endswith('.json')
])

for filename in service_files:
    filepath = os.path.join(by_service_path, filename)

    with open(filepath, 'r') as f:
        service_data = json.load(f)

    actions = service_data.get('actions', [])

    # Add to combined list
    all_documents.extend(actions)

    # Add to ground truth database
    for action in actions:
        action_key = action['action']
        ground_truth['valid_actions'].append(action_key)
        ground_truth['action_to_resources'][action_key]  = action['resource_types']
        ground_truth['action_to_conditions'][action_key] = action['condition_keys']
        ground_truth['action_details'][action_key]       = action

    print(f"  ✓ {service_data['service']:<30} {len(actions):>4} actions loaded")

# ── Save combined action_documents.json ──────────────────────────────
combined_path = os.path.join(
    DRIVE_BASE, 'data/processed/action_documents.json'
)
with open(combined_path, 'w') as f:
    json.dump(all_documents, f, indent=2)

# ── Save ground_truth_actions.json ───────────────────────────────────
gt_path = os.path.join(
    DRIVE_BASE, 'data/processed/ground_truth_actions.json'
)
with open(gt_path, 'w') as f:
    json.dump(ground_truth, f, indent=2)

print()
print(f"✓ action_documents.json    → {len(all_documents)} total actions")
print(f"✓ ground_truth_actions.json → {len(ground_truth['valid_actions'])} valid actions")
print()
print(f"Saved to: {os.path.join(DRIVE_BASE, 'data/processed/')}")

Building combined files from per-service data...

  ✓ AWS Certificate Manager          17 actions loaded
  ✓ Amazon API Gateway                3 actions loaded
  ✓ Amazon API Gateway Management    13 actions loaded
  ✓ Amazon API Gateway Management V2   36 actions loaded
  ✓ AWS Athena                       81 actions loaded
  ✓ Amazon Bedrock                  245 actions loaded
  ✓ AWS CloudFormation               94 actions loaded
  ✓ Amazon CloudWatch                62 actions loaded
  ✓ Amazon CloudWatch Logs          127 actions loaded
  ✓ AWS CodeBuild                    67 actions loaded
  ✓ AWS CodeDeploy                   48 actions loaded
  ✓ AWS CodePipeline                 44 actions loaded
  ✓ AWS CodeStar Notifications       13 actions loaded
  ✓ AWS Config                       97 actions loaded
  ✓ AWS Directory Service            91 actions loaded
  ✓ Amazon DynamoDB                  78 actions loaded
  ✓ Amazon EC2                      780 actions loaded
  ✓ Amazon EC

In [ ]:
#Scrape 500 managed policies with three-tier selection

import requests
import json
import time
import os
from bs4 import BeautifulSoup
from tqdm import tqdm
from collections import Counter

# ── Configuration ─────────────────────────────────────────────────────
CORE_SERVICES = [svc['prefix'] for svc in SERVICES_CONFIG]

def score_policy_relevance(policy_name):
    """
    Score a policy by relevance to core IAM use cases.
    """
    name_lower = policy_name.lower()
    score      = 0

    high_value_keywords = {
        's3': 15, 'lambda': 15, 'dynamodb': 15,
        'ec2': 15, 'iam': 15, 'rds': 15,
        'sns': 15, 'sqs': 15, 'cloudwatch': 15,
        'cloudformation': 15, 'logs': 12,
        'kms': 12, 'sts': 12, 'secretsmanager': 12,
        'ecr': 12, 'ecs': 12, 'ssm': 12, 'xray': 12
    }

    medium_value_keywords = {
        'readonly':      8,
        'fullaccess':    8,
        'poweruser':     8,
        'administrator': 6,
        'securityaudit': 6,
        'viewonly':      6,
        'execution':     5,
        'managed':       3
    }

    for keyword, points in {
        **high_value_keywords,
        **medium_value_keywords
    }.items():
        if keyword in name_lower:
            score += points

    return score


def get_policy_index():
    """Fetch all managed policy (name, url) pairs from index page."""
    INDEX_URL = (
        "https://docs.aws.amazon.com/aws-managed-policy/"
        "latest/reference/policy-list.html"
    )
    BASE    = (
        "https://docs.aws.amazon.com"
        "/aws-managed-policy/latest/reference/"
    )
    headers = {'User-Agent': 'Mozilla/5.0'}

    response = requests.get(INDEX_URL, headers=headers)
    soup     = BeautifulSoup(response.content, 'html.parser')

    links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if href.endswith('.html') and not href.startswith('http'):
            name = link.text.strip()
            if name:
                links.append((name, BASE + href))

    return links


def scrape_single_policy(policy_name, policy_url):
    """Scrape a single managed policy page."""
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        resp = requests.get(policy_url, headers=headers, timeout=10)
        page = BeautifulSoup(resp.content, 'html.parser')

        policy_json = None

        # Primary: <pre> tag
        pre_block = page.find('pre')
        if pre_block:
            try:
                policy_json = json.loads(pre_block.text.strip())
            except json.JSONDecodeError:
                pass

        # Fallback: <code> tag containing JSON
        if not policy_json:
            for code_tag in page.find_all('code'):
                text = code_tag.text.strip()
                if text.startswith('{') and 'Version' in text:
                    try:
                        policy_json = json.loads(text)
                        break
                    except json.JSONDecodeError:
                        continue

        if not policy_json or 'Statement' not in policy_json:
            return None

        # Extract and deduplicate actions
        actions_used = []
        for statement in policy_json.get('Statement', []):
            raw = statement.get('Action', [])
            if isinstance(raw, str):
                raw = [raw]
            actions_used.extend(raw)

        return {
            'policy_name':     policy_name,
            'url':             policy_url,
            'policy_document': policy_json,
            'actions_used':    list(set(actions_used)),  # deduplicated
            'relevance_score': score_policy_relevance(policy_name),
            'full_text': (
                f"AWS Managed Policy: {policy_name}. "
                f"Actions granted: "
                f"{', '.join(list(set(actions_used))[:20])}."
            )
        }

    except Exception:
        return None


def scrape_managed_policies_500(target=500,
                                 tier2_min_score=5,
                                 tier3_per_service=5):
    """
    Scrape up to target managed policies using three-tier selection.

    Tier 1: top 200 by relevance score
            high quality, directly relevant

    Tier 2: next policies with score >= tier2_min_score
            medium quality, filters noise

    Tier 3: guaranteed coverage per core service
            ensures no core service is underrepresented
    """

    print("Fetching policy index...")
    all_links = get_policy_index()
    print(f"Total policies available: {len(all_links)}")
    print()

    # Score all policies
    scored = [
        (name, url, score_policy_relevance(name))
        for name, url in all_links
    ]
    scored.sort(key=lambda x: x[2], reverse=True)

    # ── Tier 1: top 200 ───────────────────────────────────────────────
    tier1 = [(name, url) for name, url, score in scored[:200]]
    print(f"Tier 1 (top 200 by score):        {len(tier1)} policies")

    # ── Tier 2: next policies with score >= 5 ─────────────────────────
    tier1_names = {name for name, _ in tier1}
    tier2 = [
        (name, url) for name, url, score in scored[200:]
        if score >= tier2_min_score
        and name not in tier1_names
    ]
    print(f"Tier 2 (score>={tier2_min_score}, after tier1): "
          f"{len(tier2)} policies")

    # ── Tier 3: guaranteed coverage per core service ──────────────────
    tier1_tier2_names = tier1_names | {name for name, _ in tier2}
    tier3 = []

    print(f"Tier 3 (guaranteed coverage):")
    for service in CORE_SERVICES:
        service_policies = [
            (name, url) for name, url, score in scored
            if service.lower() in name.lower()
            and name not in tier1_tier2_names
        ]
        added = 0
        for name, url in service_policies[:tier3_per_service]:
            tier3.append((name, url))
            added += 1
        if added > 0:
            print(f"  + {added} extra policies for {service}")

    # ── Combine and deduplicate ───────────────────────────────────────
    all_selected = tier1 + tier2 + tier3
    seen         = set()
    deduped      = []
    for name, url in all_selected:
        if name not in seen:
            seen.add(name)
            deduped.append((name, url))

    # Cap at target
    final = deduped[:target]

    print()
    print(f"Total unique policies selected: {len(deduped)}")
    print(f"Final count (capped at {target}): {len(final)}")
    print()

    # Show top 20 for verification
    print("Top 20 selected policies by score:")
    for name, url in final[:20]:
        score = score_policy_relevance(name)
        print(f"  score={score:>3}  {name}")
    print()

    # ── Scrape selected policies ──────────────────────────────────────
    policies = []
    failed   = []

    for policy_name, policy_url in tqdm(final):
        result = scrape_single_policy(policy_name, policy_url)

        if result:
            policies.append(result)
        else:
            failed.append(policy_name)

        time.sleep(0.5)

    print()
    print(f"Successfully scraped: {len(policies)} policies")
    if failed:
        print(f"Failed:               {len(failed)} policies")
        for name in failed[:5]:
            print(f"  ✗ {name}")

    return policies


# ── Run ───────────────────────────────────────────────────────────────
print("=" * 55)
print("SCRAPING 500 MANAGED POLICIES (THREE-TIER SELECTION)")
print("=" * 55)
print()

managed_policies = scrape_managed_policies_500(
    target=500,
    tier2_min_score=5,
    tier3_per_service=5
)

# ── Save ──────────────────────────────────────────────────────────────
raw_path = os.path.join(
    DRIVE_BASE,
    'data/raw/managed_policies/managed_policies_raw.json'
)
processed_path = os.path.join(
    DRIVE_BASE, 'data/processed/managed_policies.json'
)

with open(raw_path, 'w') as f:
    json.dump(managed_policies, f, indent=2)
with open(processed_path, 'w') as f:
    json.dump(managed_policies, f, indent=2)

print(f"\n✓ Saved {len(managed_policies)} managed policies")

# ── Verify service coverage ───────────────────────────────────────────
print()
print("Core service coverage:")
service_coverage = Counter()
for policy in managed_policies:
    for action in policy['actions_used']:
        if ':' in action:
            service = action.split(':')[0]
            service_coverage[service] += 1

for service in CORE_SERVICES:
    count  = service_coverage.get(service, 0)
    status = "✓" if count > 0 else "✗ MISSING"
    print(f"  {status}  {service:<25} {count:>4} action references")

print()
print("Top 20 services by action references:")
for service, count in service_coverage.most_common(20):
    marker = "★" if service in CORE_SERVICES else " "
    print(f"  {marker} {service:<30} {count:>4} references")

SCRAPING 500 MANAGED POLICIES (THREE-TIER SELECTION)

Fetching policy index...
Total policies available: 1477

Tier 1 (top 200 by score):        200 policies
Tier 2 (score>=5, after tier1): 600 policies
Tier 3 (guaranteed coverage):
  + 5 extra policies for sso

Total unique policies selected: 805
Final count (capped at 500): 500

Top 20 selected policies by score:
  score= 38  CloudWatchOpenSearchDashboardsFullAccess
  score= 35  AmazonS3ObjectLambdaExecutionRolePolicy
  score= 35  AmazonS3OutpostsFullAccess
  score= 35  AmazonS3OutpostsReadOnlyAccess
  score= 35  AWSLambdaDynamoDBExecutionRole
  score= 35  AWSLambdaSQSQueueExecutionRole
  score= 35  CloudWatchLambdaApplicationSignalsExecutionRolePolicy
  score= 35  CloudWatchLambdaInsightsExecutionRolePolicy
  score= 35  CloudWatchLogsFullAccess
  score= 35  CloudWatchLogsReadOnlyAccess
  score= 33  AWSLambdaManagedEC2ResourceOperator
  score= 32  AWSSecretsManagerClientReadOnlyAccess
  score= 30  AmazonSSMManagedEC2InstanceDefaultPo

100%|██████████| 500/500 [05:28<00:00,  1.52it/s]



Successfully scraped: 500 policies

✓ Saved 500 managed policies

Core service coverage:
  ✓  s3                         372 action references
  ✓  lambda                     118 action references
  ✓  dynamodb                    90 action references
  ✓  ec2                       1657 action references
  ✓  iam                        433 action references
  ✓  rds                        209 action references
  ✓  sns                        139 action references
  ✓  sqs                         61 action references
  ✓  cloudwatch                 297 action references
  ✓  cloudformation             145 action references
  ✓  access-analyzer              8 action references
  ✓  sso                          3 action references
  ✓  logs                       386 action references
  ✓  kms                        199 action references
  ✓  sts                         27 action references
  ✓  secretsmanager             187 action references
  ✓  ecr                        129 action ref

In [ ]:
# Cell 13: Verify complete data collection

print("=" * 55)
print("DATA COLLECTION COMPLETE — FINAL VERIFICATION")
print("=" * 55)

# ── Check per-service files ───────────────────────────────────────────
print("\nPer-service files (data/processed/by_service/):")
by_service_path = os.path.join(DRIVE_BASE, 'data/processed/by_service/')

for filename in sorted(os.listdir(by_service_path)):
    if not filename.endswith('.json'):
        continue
    filepath = os.path.join(by_service_path, filename)
    with open(filepath) as f:
        data = json.load(f)
    size_kb = os.path.getsize(filepath) / 1024
    count   = data.get('action_count', 0)
    print(f"  ✓ {filename:<35} {count:>4} actions  {size_kb:>6.1f} KB")

# ── Check combined files ──────────────────────────────────────────────
print("\nCombined files (data/processed/):")

combined_files = {
    'action_documents.json':     'Total actions (all services combined)',
    'ground_truth_actions.json': 'Valid actions in ground truth database',
    'managed_policies.json':     'Managed policies',
}

for filename, description in combined_files.items():
    filepath = os.path.join(DRIVE_BASE, 'data/processed/', filename)
    if os.path.exists(filepath):
        with open(filepath) as f:
            data = json.load(f)
        size_kb = os.path.getsize(filepath) / 1024
        count   = (
            len(data) if isinstance(data, list)
            else len(data.get('valid_actions', []))
        )
        print(f"  ✓ {filename:<40} {count:>4} items  {size_kb:>6.1f} KB")
    else:
        print(f"  ✗ {filename} — NOT FOUND")

# ── Print final folder tree ───────────────────────────────────────────
print("\nFinal folder structure:")
print()
for root, dirs, files in os.walk(os.path.join(DRIVE_BASE, 'data')):
    dirs[:] = [d for d in sorted(dirs) if not d.startswith('.')]
    depth   = root.replace(DRIVE_BASE, '').count(os.sep)
    indent  = '    ' * depth
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = '    ' * (depth + 1)
    for file in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{sub_indent}📄 {file} ({size_kb:.1f} KB)")

# ── Show one sample document ──────────────────────────────────────────
print("\nSample action document (s3:GetObject):")
print()
with open(os.path.join(DRIVE_BASE, 'data/processed/by_service/s3_actions.json')) as f:
    s3_data = json.load(f)

sample = next(
    (a for a in s3_data['actions'] if a['action'] == 's3:GetObject'),
    s3_data['actions'][0]
)
print(json.dumps(sample, indent=2))

DATA COLLECTION COMPLETE — FINAL VERIFICATION

Per-service files (data/processed/by_service/):
  ✓ acm_actions.json                      17 actions    12.7 KB
  ✓ apigateway_actions.json                3 actions     2.2 KB
  ✓ apigateway_management_actions.json    13 actions    11.0 KB
  ✓ apigateway_management_v2_actions.json   36 actions    26.1 KB
  ✓ athena_actions.json                   81 actions    56.3 KB
  ✓ bedrock_actions.json                 245 actions   177.3 KB
  ✓ cloudformation_actions.json           94 actions    74.3 KB
  ✓ cloudwatch_actions.json               62 actions    45.6 KB
  ✓ cloudwatch_logs_actions.json         127 actions    92.8 KB
  ✓ codebuild_actions.json                67 actions    50.1 KB
  ✓ codedeploy_actions.json               48 actions    38.1 KB
  ✓ codepipeline_actions.json             44 actions    33.1 KB
  ✓ codestar_notifications_actions.json   13 actions    10.5 KB
  ✓ config_actions.json                   97 actions    79.3 KB
  ✓ ds_